# Vision in Action (ViA):Demo Inference on the Bag task


## 1. Environment check

In [ ]:
import sys, time
from pathlib import Path
import torch

PART2 = Path.cwd()
assert (PART2 / "via_tools").is_dir(), "start Jupyter in Part2/ (the folder that contains via_tools/)"
sys.path.insert(0, str(PART2))
from via_tools import DEFAULT_DATA_DIR, DEFAULT_CKPT_DIR, DEFAULT_OUT_DIR, DEFAULT_CONFIG, DEFAULT_CKPT_NAME

DATA_DIR, CKPT_DIR, OUT_DIR = DEFAULT_DATA_DIR, DEFAULT_CKPT_DIR, DEFAULT_OUT_DIR
RUN_TRAIN_SPLIT = True   # supplementary eval on the 150 training episodes (~1 min)
OUT_DIR.mkdir(exist_ok=True)

import diffusers, zarr, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| zarr", zarr.__version__, "| numpy", numpy.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 2**30:.1f} GB")
else:
    print("No CUDA GPU found: evaluation will run on the CPU (slow).")

required = {DATA_DIR / "bag_train.zarr.zip": "official data Drive folder",
            DATA_DIR / "bag_val.zarr.zip": "official data Drive folder",
            CKPT_DIR / DEFAULT_CKPT_NAME: "official checkpoint Drive folder",
            DEFAULT_CONFIG: "official repo (git clone into Part2/external/vision-in-action)"}
missing = [p for p in required if not p.exists()]
for p, src in required.items():
    print(("ok      " if p.exists() else "MISSING ") + str(p.relative_to(PART2)) + ("" if p.exists() else f"   <- download from the {src}"))
assert not missing, "put the missing files where shown above (see README.md)"


## 2. Data inspection (CPU)

Reads only `proprio`, `action` and `episode_ends` from the 13.6 GB train zip (train-set min/max stats, cached in `output/data/stats_bag.npz`); images are read lazily as uint8.

In [ ]:
from IPython.display import Video, display, Markdown
from via_tools import data

summary = data.inspect(DATA_DIR, OUT_DIR)
display(Video(str(OUT_DIR / "videos" / "raw_episode.mp4"), embed=True, width=448))

## 3. Open-loop evaluation (GPU)

Two metrics, per step and averaged over the episode:
- **action MSE**: the released code's `val_epoch_action_error` (raw units, all 23 dims);
- **normalised action MSE** (this work): the error after mapping every dim to [-1, 1] with the training min/max, with predicted quaternions sign-aligned to the human ones; over all dims and per part (head, arm pose, gripper).

First on the val split (1 held-out episode).

In [ ]:
from via_tools import eval_offline

t = time.time()
val_res = eval_offline.evaluate(data_dir=DATA_DIR, ckpt_dir=CKPT_DIR, out_dir=OUT_DIR,
                                split="val", name="bag", n_samples=3, seed=0)
print(f"{time.time() - t:.0f} s")


Supplementary: the same evaluation on the 150 **training** episodes (seen during training, so this measures fit, not generalisation; it is used for the best/worst episodes).

In [ ]:
if RUN_TRAIN_SPLIT:
    t = time.time()
    train_res = eval_offline.evaluate(data_dir=DATA_DIR, ckpt_dir=CKPT_DIR, out_dir=OUT_DIR,
                                      split="train", name="bag_train", n_samples=3, seed=0)
    print(f"{time.time() - t:.0f} s")


## 4. Videos (CPU)

In [ ]:
from via_tools import visualize

t = time.time()
files = visualize.make_videos("bag", data_dir=DATA_DIR, out_dir=OUT_DIR)
if RUN_TRAIN_SPLIT:
    files += visualize.make_videos("bag_train", data_dir=DATA_DIR, out_dir=OUT_DIR)
print(f"{time.time() - t:.0f} s")

In [ ]:
display(Video(str(OUT_DIR / "videos" / "video_bag_ep0.mp4"), embed=True, width=960))

## 5. Results

In [ ]:
from via_tools import report

display(Markdown("**Held-out val episode**"), report.results_table(val_res))
if RUN_TRAIN_SPLIT:
    display(Markdown("**150 training episodes (seen in training)**"), report.results_table(train_res))
    display(Markdown("**Train: best / worst episodes by normalised MSE**"), report.best_worst_table(OUT_DIR, "bag_train"))
display(Markdown("**Why a normalised MSE: share of each dim group**"), report.mse_variants_table(OUT_DIR))